# Эффект января

Проверка второй гипотезы: отличается ли доходность в отдельные
месяцы. Классическая формулировка «эффекта января» — акции
растут в первый месяц года сильнее обычного.

Данные и подход те же, что в 02_analysis.ipynb.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"

prices = pd.read_csv(RAW / "moex_prices.csv", parse_dates=["TRADEDATE"])

prices = prices[prices["CLOSE"].notna()]
prices = prices[["TRADEDATE", "SECID", "CLOSE"]]
prices = prices.sort_values(["SECID", "TRADEDATE"]).reset_index(drop=True)

prices["RETURN"] = prices.groupby("SECID")["CLOSE"].pct_change()
prices.loc[prices["RETURN"].abs() > 0.5, "RETURN"] = None
prices = prices[prices["TRADEDATE"].dt.dayofweek < 5]

daily = prices.groupby("TRADEDATE")["RETURN"].mean().reset_index()
daily["MONTH"] = daily["TRADEDATE"].dt.month
daily["YEAR"] = daily["TRADEDATE"].dt.year

print(daily.shape)
daily.head()

(3160, 4)


,TRADEDATE,RETURN,MONTH,YEAR
0,2014-01-06,NaN,1,2014
1,2014-01-08,-0.000093,1,2014
2,2014-01-09,-0.001704,1,2014
3,2014-01-10,-0.000401,1,2014
4,2014-01-13,0.006905,1,2014


In [2]:
MONTH_NAMES = ["Янв", "Фев", "Мар", "Апр", "Май", "Июн",
               "Июл", "Авг", "Сен", "Окт", "Ноя", "Дек"]

by_month = daily.groupby("MONTH")["RETURN"].agg(["count", "mean", "std"])
by_month.index = MONTH_NAMES

by_month["mean_pct"] = by_month["mean"] * 100
by_month["se_pct"] = by_month["std"] / by_month["count"]**0.5 * 100

by_month.round(4)

,count,mean,std,mean_pct,se_pct
Янв,256,0.0017,0.0116,0.1743,0.0726
Фев,251,-0.0009,0.0220,-0.0873,0.1388
Мар,260,0.0003,0.0186,0.0285,0.1152
Апр,279,0.0005,0.0137,0.0456,0.0819
Май,258,-0.0002,0.0109,-0.0197,0.0676
Июн,265,0.0001,0.0110,0.0098,0.0673
Июл,288,0.0004,0.0127,0.0365,0.0747
Авг,275,0.0008,0.0106,0.0773,0.0637
Сен,258,-0.0006,0.0134,-0.0621,0.0834
Окт,266,0.0001,0.0118,0.0087,0.0725


In [3]:
jan = daily.loc[daily["MONTH"] == 1, "RETURN"].dropna()
rest = daily.loc[daily["MONTH"] != 1, "RETURN"].dropna()

t_stat, p_value = stats.ttest_ind(jan, rest, equal_var=False)

print(f"январь:    {jan.mean()*100:+.4f}%  (n={len(jan)})")
print(f"остальные: {rest.mean()*100:+.4f}%  (n={len(rest)})")
print(f"t = {t_stat:.3f}, p = {p_value:.4f}")

январь:    +0.1743%  (n=256)
остальные: +0.0227%  (n=2903)
t = 1.969, p = 0.0498


In [4]:
jan_by_year = daily[daily["MONTH"] == 1].groupby("YEAR")["RETURN"].mean() * 100
print(jan_by_year.round(3))
print(f"\nположительных лет: {(jan_by_year > 0).sum()} из {len(jan_by_year)}")

YEAR
2014    0.050
2015    0.917
2016    0.168
2017    0.108
2018    0.310
2019    0.220
2020    0.129
2021   -0.031
2022   -0.272
2023    0.225
2024    0.227
2025    0.102
2026    0.122
Name: RETURN, dtype: float64

положительных лет: 11 из 13


In [5]:
print(jan_by_year)
print()
print(f"среднее по годам:        {jan_by_year.mean():+.4f}%")
print(f"без 2015:                {jan_by_year.drop(2015).mean():+.4f}%")
print(f"медиана по годам:        {jan_by_year.median():+.4f}%")

jan_no2015 = daily[(daily["MONTH"] == 1) & (daily["YEAR"] != 2015)]["RETURN"].dropna()
rest_no2015 = daily[(daily["MONTH"] != 1) & (daily["YEAR"] != 2015)]["RETURN"].dropna()

t2, p2 = stats.ttest_ind(jan_no2015, rest_no2015, equal_var=False)
print(f"\nбез 2015: t = {t2:.3f}, p = {p2:.4f}")

YEAR
2014    0.049758
2015    0.917181
2016    0.167565
2017    0.107952
2018    0.310044
2019    0.219893
2020    0.128813
2021   -0.030543
2022   -0.271764
2023    0.225382
2024    0.227053
2025    0.102413
2026    0.121731
Name: RETURN, dtype: float64

среднее по годам:        +0.1750%
без 2015:                +0.1132%
медиана по годам:        +0.1288%

без 2015: t = 1.248, p = 0.2131


### Результат: эффект января

Январь — лучший месяц по средней дневной доходности (+0,174%
против +0,023% у остальных), t = 1,97, p = 0,0498 — ровно
на границе значимости.

Гипотеза была заявлена заранее (в README), поэтому поправка
на множественные сравнения не требуется.

Проверка на устойчивость: январь 2015 дал +0,917% в день —
вчетверо больше любого другого января в выборке.

Без 2015 года:
- среднее падает с 0,175% до 0,113%
- p вырастает с 0,0498 до 0,2131

Медиана по годам (+0,129%) ниже среднего (+0,175%) —
признак влияния выброса.

Вывод: эффект января на этих данных не подтверждается.
Пограничная значимость в полной выборке держится на одном
аномальном январе.

Февраль формально худший месяц (-0,087%): стандартная ошибка 0,139 п.п. в полтора раза больше самого значения, интервал уверенно накрывает ноль. Разброс внутри февраля (std 0,0220) вдвое выше, чем у большинства месяцев.